# preQCA — Preparar los datos para fsQCA

QCA necesita representar las condiciones y el resultado como **pertenencias a conjuntos**.

Este trabajo analiza **10 gobiernos presidenciales de Perú, Chile y Colombia** para explorar qué configuraciones de condiciones políticas se relacionan con una mayor **inestabilidad del gabinete**.

Las mediciones originales corresponden a cuatro dimensiones del gobierno: el margen de victoria en el balotaje, el respaldo legislativo propio del presidente, el número de partidos o listas de oposición y la rotación ministerial. Las tres primeras serán utilizadas como **condiciones** y la rotación ministerial como **resultado (outcome)**.

En este notebook trabajaremos en paralelo con la guía: conoceremos las mediciones originales, definiremos conceptualmente los conjuntos, estableceremos los umbrales, calibraremos las mediciones, compararemos los valores originales con las membresías y finalmente guardaremos la base preparada para el análisis fsQCA.

**Unidad de análisis:** cada gobierno presidencial.

`mediciones originales → conceptualización → umbrales → calibración → membresías`


## Instalar la librería

En Google Colab necesitamos instalar `setqca` una vez por sesión.


In [1]:
!pip -q install setqca


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 746.8 kB/s eta 0:00:00


## preQCA-01: conocer las mediciones originales

Cada fila representa un **gobierno presidencial** de Perú, Chile o Colombia.

Las variables todavía son **mediciones empíricas**, no membresías QCA.

Antes de calibrarlas, se establece para cada variable su definición conceptual, unidad de medición, fuente, rango y temporalidad. Las condiciones corresponden a características observables al inicio o antes del gobierno, mientras que la rotación ministerial corresponde al período efectivo de cada gobierno y constituye el resultado (outcome).

#### **Diccionario de variables**

##### `margen_balotaje_pp`
Diferencia, en puntos porcentuales, entre el porcentaje de votos del candidato ganador y el candidato que quedó segundo en la segunda vuelta. Un valor bajo significa que la elección fue más ajustada; un valor alto, que el ganador tuvo una ventaja mayor.

##### `porcentaje_escanos_gobierno`
Porcentaje de escaños de la cámara de referencia que tenía la lista o pacto electoral del presidente al inicio de su gobierno. Se calcula dividiendo los escaños de esa lista o pacto entre el total de escaños de la cámara y multiplicando por 100.

##### `n_partidos_oposicion`
Número de partidos o listas de oposición que tenían representación en la cámara de referencia al inicio del gobierno. Cada partido o lista con al menos un escaño cuenta como una unidad.

##### `rotacion_ministerial_por_anio`
Número promedio de cambios de ministros por cada año de gobierno. Se calcula dividiendo el total de cambios de ministros entre la duración del gobierno en años. Se cuentan los cambios independientemente de si fueron por renuncia, remoción, reemplazo u otra causa.

In [2]:
import pandas as pd
from setqca import calibrate_direct
from google.colab import drive

# Conectar Google Drive
drive.mount('/content/drive')

# Leer la base preQCA
ruta = '/content/drive/MyDrive/DOCSm/Gary/data/data_preQCA.xlsx'
qca_raw = pd.read_excel(ruta)

qca_raw.head()


Mounted at /content/drive


,pais,presidente,margen_balotaje_pp,porcentaje_escanos_gobierno,n_partidos_oposicion,rotacion_ministerial_por_anio
0,Perú,Ollanta Humala,2.898,36.154,5,11
1,Perú,Pedro Pablo Kuczynski,0.240,13.846,5,15
2,Perú,Pedro Castillo,0.252,28.462,9,46
3,Chile,Sebastián Piñera,3.220,44.700,3,7
4,Chile,Michelle Bachelet,24.340,15.789,3,6


In [3]:
qca_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   pais                           10 non-null     object 
 1   presidente                     10 non-null     object 
 2   margen_balotaje_pp             10 non-null     float64
 3   porcentaje_escanos_gobierno    10 non-null     float64
 4   n_partidos_oposicion           10 non-null     int64  
 5   rotacion_ministerial_por_anio  10 non-null     int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 612.0+ bytes


## preQCA-02: De las mediciones a los conjuntos

Las mediciones originales no constituyen todavía conjuntos QCA.
Primero se define conceptualmente qué significa que un caso pertenezca
al conjunto de interés. La calibración posterior traducirá cada medición
original a un grado de pertenencia entre 0 y 1.

Para este estudio se construirán cuatro conjuntos:


### 1. BALOTAJE_ESTRECHO

Conjunto de gobiernos que llegaron al poder mediante una segunda vuelta presidencial estrecha.

La medición utilizada es `margen_balotaje_pp`.

Valores bajos del margen indican una elección más ajustada y, por tanto, mayor pertenencia al conjunto `BALOTAJE_ESTRECHO`.

---

### 2. BAJO_APOYO_LEGISLATIVO

Conjunto de gobiernos cuyo partido o pacto presidencial posee una baja proporción de escaños en la cámara de referencia al inicio del gobierno.

La medición utilizada es `porcentaje_escanos_gobierno`.

Valores bajos indican menor respaldo legislativo propio y, por tanto, mayor pertenencia al conjunto `BAJO_APOYO_LEGISLATIVO`.

---

### 3. OPOSICION_FRAGMENTADA

Conjunto de gobiernos que enfrentan un mayor número de partidos o listas de oposición con representación en la cámara de referencia.

La medición utilizada es `n_partidos_oposicion`.

Valores altos indican una oposición más numerosa y, por tanto, mayor pertenencia al conjunto `OPOSICION_FRAGMENTADA`.

---

### 4. ALTA_ROTACION_MINISTERIAL

Conjunto de gobiernos caracterizados por una alta rotación de ministros durante su período de gobierno.

La medición utilizada es `rotacion_ministerial_por_anio`.

Valores altos indican mayor rotación y, por tanto, mayor pertenencia al conjunto `ALTA_ROTACION_MINISTERIAL`.


In [4]:
conjuntos = {
    "BALOTAJE_ESTRECHO": "margen_balotaje_pp",
    "BAJO_APOYO_LEGISLATIVO": "porcentaje_escanos_gobierno",
    "OPOSICION_FRAGMENTADA": "n_partidos_oposicion",
    "ALTA_ROTACION_MINISTERIAL": "rotacion_ministerial_por_anio"
}

conjuntos


{'BALOTAJE_ESTRECHO': 'margen_balotaje_pp',
 'BAJO_APOYO_LEGISLATIVO': 'porcentaje_escanos_gobierno',
 'OPOSICION_FRAGMENTADA': 'n_partidos_oposicion',
 'ALTA_ROTACION_MINISTERIAL': 'rotacion_ministerial_por_anio'}

### Justificación de los umbrales

**BALOTAJE_ESTRECHO — (15, 5, 1)**  
Se considera que un margen de hasta 1 pp representa una elección muy estrecha y, por tanto, pertenencia plena al conjunto. Un margen de 5 pp constituye el punto de máxima ambigüedad, mientras que 15 pp representa una victoria suficientemente amplia para quedar fuera del conjunto.

**BAJO_APOYO_LEGISLATIVO — (50, 25, 10)**  
Una proporción de 10% o menos de escaños representa un bajo apoyo legislativo propio y, por tanto, pertenencia plena al conjunto. El 25% constituye el punto de máxima ambigüedad, mientras que 50% representa una proporción alta de escaños del oficialismo y, por tanto, pertenencia plena al conjunto contrario.

**OPOSICION_FRAGMENTADA — (2, 6, 8)**  
La presencia de 2 partidos o listas de oposición representa una oposición poco fragmentada y, por tanto, baja pertenencia al conjunto. 5 unidades constituye el punto de máxima ambigüedad, mientras que 8 o más representan una oposición con un mayor número de fuerzas y, por tanto, pertenencia plena al conjunto.

**ALTA_ROTACION_MINISTERIAL — (5, 10, 20)**  
Una tasa de 5 cambios ministeriales por año o menos representa una rotación baja y, por tanto, baja pertenencia al conjunto. 10 cambios por año constituye el punto de máxima ambigüedad, mientras que 20 o más representa una rotación ministerial alta y, por tanto, pertenencia plena al conjunto.

In [5]:
umbrales = {

    "BALOTAJE_ESTRECHO": (15, 5, 1),

    "BAJO_APOYO_LEGISLATIVO": (50, 25, 10),

    "OPOSICION_FRAGMENTADA": (2, 6, 8),

    "ALTA_ROTACION_MINISTERIAL": (5, 10, 20)

}

umbrales

{'BALOTAJE_ESTRECHO': (15, 5, 1),
 'BAJO_APOYO_LEGISLATIVO': (50, 25, 10),
 'OPOSICION_FRAGMENTADA': (2, 6, 8),
 'ALTA_ROTACION_MINISTERIAL': (5, 10, 20)}

### preQCA-03: Calibración de las mediciones

`calibrate_direct()` recibe una medición original y tres referencias sustantivas para definir su pertenencia al conjunto:

- **`full_out`**: valor que representa pertenencia plena al conjunto contrario, es decir, el caso está claramente fuera del conjunto.
- **`crossover`**: punto de máxima ambigüedad entre pertenecer y no pertenecer al conjunto. Corresponde a una pertenencia de 0.5.
- **`full_in`**: valor que representa pertenencia plena al conjunto, es decir, el caso está claramente dentro.

A partir de estos tres puntos, `calibrate_direct()` transforma cada medición original en un grado de pertenencia entre 0 y 1.

La calibración se realiza por separado para cada conjunto, utilizando sus propios umbrales. En `BALOTAJE_ESTRECHO` y `BALOTAJE_APOYO_LEGISLATIVO`, la relación es inversa: valores menores del margen implican una mayor pertenencia al conjunto.


In [6]:
qca = pd.DataFrame({
    "pais": qca_raw["pais"],
    "presidente": qca_raw["presidente"],

    "BALOTAJE_ESTRECHO": calibrate_direct(
        qca_raw["margen_balotaje_pp"],
        full_out=15,
        crossover=5,
        full_in=1
    ),

    "BAJO_APOYO_LEGISLATIVO": calibrate_direct(
        qca_raw["porcentaje_escanos_gobierno"],
        full_out=50,
        crossover=25,
        full_in=10
    ),

    "OPOSICION_FRAGMENTADA": calibrate_direct(
        qca_raw["n_partidos_oposicion"],
        full_out=2,
        crossover=6,
        full_in=8
    ),

    "ALTA_ROTACION_MINISTERIAL": calibrate_direct(
        qca_raw["rotacion_ministerial_por_anio"],
        full_out=5,
        crossover=10,
        full_in=20
    )
})

qca.head()

,pais,presidente,BALOTAJE_ESTRECHO,BAJO_APOYO_LEGISLATIVO,OPOSICION_FRAGMENTADA,ALTA_ROTACION_MINISTERIAL
0,Perú,Ollanta Humala,0.824524,0.211870,0.323855,0.573084
1,Perú,Pedro Pablo Kuczynski,0.970798,0.899301,0.323855,0.813395
2,Perú,Pedro Castillo,0.970547,0.399453,0.988070,0.999975
3,Chile,Sebastián Piñera,0.787559,0.089462,0.099005,0.145958
4,Chile,Michelle Bachelet,0.003353,0.859130,0.099005,0.086626


## preQCA-04: Comparar mediciones y membresías

Ahora observamos lado a lado las **mediciones originales** y las **membresías calibradas**.

`medición original → pertenencia al conjunto`

Esta comparación permite verificar cómo cada valor de la medición original se transforma en su grado de pertenencia al conjunto correspondiente.

In [7]:
comparacion = pd.DataFrame({

    "pais": qca_raw["pais"],
    "presidente": qca_raw["presidente"],

    "margen_balotaje_pp": qca_raw["margen_balotaje_pp"],
    "BALOTAJE_ESTRECHO": qca["BALOTAJE_ESTRECHO"],

    "porcentaje_escanos_gobierno": qca_raw["porcentaje_escanos_gobierno"],
    "BAJO_APOYO_LEGISLATIVO": qca["BAJO_APOYO_LEGISLATIVO"],

    "n_partidos_oposicion": qca_raw["n_partidos_oposicion"],
    "OPOSICION_FRAGMENTADA": qca["OPOSICION_FRAGMENTADA"],

    "rotacion_ministerial_por_anio": qca_raw["rotacion_ministerial_por_anio"],
    "ALTA_ROTACION_MINISTERIAL": qca["ALTA_ROTACION_MINISTERIAL"]
})

comparacion

,pais,presidente,margen_balotaje_pp,BALOTAJE_ESTRECHO,porcentaje_escanos_gobierno,BAJO_APOYO_LEGISLATIVO,n_partidos_oposicion,OPOSICION_FRAGMENTADA,rotacion_ministerial_por_anio,ALTA_ROTACION_MINISTERIAL
0,Perú,Ollanta Humala,2.898,0.824524,36.154,0.211870,5,0.323855,11,0.573084
1,Perú,Pedro Pablo Kuczynski,0.240,0.970798,13.846,0.899301,5,0.323855,15,0.813395
2,Perú,Pedro Castillo,0.252,0.970547,28.462,0.399453,9,0.988070,46,0.999975
3,Chile,Sebastián Piñera,3.220,0.787559,44.700,0.089462,3,0.099005,7,0.145958
4,Chile,Michelle Bachelet,24.340,0.003353,15.789,0.859130,3,0.099005,6,0.086626
5,Chile,Sebastián Piñera,9.160,0.227078,44.100,0.095388,5,0.323855,10,0.500000
6,Chile,Gabriel Boric,11.730,0.121148,10.000,0.950000,8,0.950000,7,0.145958
7,Colombia,Juan Manuel Santos,43.126,0.000013,27.451,0.428329,9,0.988070,6,0.086626
8,Colombia,Juan Manuel Santos,6.240,0.409723,20.588,0.703925,9,0.988070,5,0.050000
9,Colombia,Iván Duque,12.797,0.091473,17.593,0.810608,12,0.999854,6,0.086626


## preQCA-05: guardar la base preparada

Una vez revisada la calibración, guardamos las membresías en un archivo separado.

Este archivo será la entrada del notebook `QCA`.


In [8]:
ruta_salida = "/content/drive/MyDrive/DOCSm/Gary/data/qca_calibrada.csv"

qca.to_csv(ruta_salida, index=False)

print("Archivo creado:", ruta_salida)
print("Casos:", len(qca))

Archivo creado: /content/drive/MyDrive/DOCSm/Gary/data/qca_calibrada.csv
Casos: 10
